# Macaque Basal Ganglia: Resting State

A NEST model of the macaque basal ganglia, built from eight nodes and validated
against published resting-state firing rates.

```
SimConfig ──────────┐
NeuronParams ───────┤
ConnectivityParams ─┼──→ Network ──→ RestingState ──→ RestingAnalysis
StriatumParams ─────┤                     (4 outputs)
InputParams ────────┘
```

Five nodes carry parameters, one builds the network, one simulates it, and one checks
the result against the literature. The parameter nodes have no inputs, so each is an
independent place to change one aspect of the model - cell properties, connectivity,
striatal detail, external input - without touching the rest.

This workflow was assembled in the web application; the notebook runs the same graph.

## 1. Nodes

In [ ]:
from neuroworkflow.core.workflow import WorkflowBuilder

from neuroworkflow.nodes.simulation.BG_Maq_SimConfig        import BG_Maq_SimConfig
from neuroworkflow.nodes.network.BG_Maq_NeuronParams        import BG_Maq_NeuronParams
from neuroworkflow.nodes.network.BG_Maq_ConnectivityParams  import BG_Maq_ConnectivityParams
from neuroworkflow.nodes.network.BG_Maq_StriatumParams      import BG_Maq_StriatumParams
from neuroworkflow.nodes.stimulus.BG_Maq_InputParams        import BG_Maq_InputParams
from neuroworkflow.nodes.network.BG_Maq_Network             import BG_Maq_Network
from neuroworkflow.nodes.simulation.BG_Maq_RestingState     import BG_Maq_RestingState
from neuroworkflow.nodes.analysis.BG_Maq_RestingAnalysis    import BG_Maq_RestingAnalysis

In [ ]:
SimConfig          = BG_Maq_SimConfig("SimConfig")
NeuronParams       = BG_Maq_NeuronParams("NeuronParams")
ConnectivityParams = BG_Maq_ConnectivityParams("ConnectivityParams")
StriatumParams     = BG_Maq_StriatumParams("StriatumParams")
InputParams        = BG_Maq_InputParams("InputParams")
Network            = BG_Maq_Network("Network")
RestingState       = BG_Maq_RestingState("RestingState")
RestingAnalysis    = BG_Maq_RestingAnalysis("RestingAnalysis")

# scalefactor_x / scalefactor_y scale the network down from the full model.
SimConfig.configure(scalefactor_x=0.8, scalefactor_y=0.8)

for node in (NeuronParams, ConnectivityParams, StriatumParams,
             InputParams, Network, RestingState, RestingAnalysis):
    node.configure()

## 2. The graph

`Network` takes all five parameter sets; `RestingAnalysis` takes all four of
`RestingState`'s outputs, so the rates, the spikes and the run metadata are all
available to the validation.

In [ ]:
wf = WorkflowBuilder("BG_Maq_RestingState", context={"results_path": "./results/bg_maq"})

for node in (SimConfig, NeuronParams, ConnectivityParams, StriatumParams,
             InputParams, Network, RestingState, RestingAnalysis):
    wf.add_node(node)

for source, port in (("SimConfig", "sim_config"),
                     ("NeuronParams", "neuron_params"),
                     ("ConnectivityParams", "connectivity_params"),
                     ("StriatumParams", "striatum_params"),
                     ("InputParams", "input_params")):
    wf.connect(source, port, "Network", port)

wf.connect("Network", "bg_network", "RestingState", "bg_network")

for port in ("mean_fr", "at_fr", "spikes", "run_info"):
    wf.connect("RestingState", port, "RestingAnalysis", port)

workflow = wf.build()
print(workflow)

## 3. Run

Building the network and simulating it both happen here. The analysis node prints
its table as it goes.

In [ ]:
assert workflow.execute(), "workflow failed — check the printed errors above"
print("\ndone")

## 4. What came out

`validation` holds the comparison the analysis printed, `score` the share of nuclei
inside their published range, and `figure_paths` whatever was drawn.

In [ ]:
validation = RestingAnalysis._output_ports["validation"].value
score      = RestingAnalysis._output_ports["score"].value

print(f"score: {score}\n")
for nucleus, entry in validation.items():
    print(f"  {nucleus:10} {entry}")

### Reading the result

A nucleus outside its band is not necessarily a mistake - published ranges come from
different animals, tasks and recording methods, and a scaled-down network will not
reproduce every one of them. It tells you where this model disagrees with the
literature, which is where to look first if something downstream behaves oddly.

The rates come from a single resting-state run. To ask whether a parameter *causes*
a rate, change it in the relevant node and re-run: the parameter nodes are separate
precisely so one aspect can move at a time.

### Re-running

Give each run its own `results_path`. Rebuilding a network into a directory already
used in this session fails on a file the previous run still holds open:

```python
for node in workflow.nodes.values():
    node._context["results_path"] = "./results/bg_maq_variant"
```